# 🧩 Semantic Change Detection (SCD) — not just *whether* it changed

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SyedRafiAhmedGIS/AnyDisasterMapping/blob/main/%URL%)
[![GitHub](https://img.shields.io/badge/Repo-AnyDisasterMapping-181717?logo=github)](https://github.com/SyedRafiAhmedGIS/AnyDisasterMapping)
![Auto](https://img.shields.io/badge/runs-automatically-3fb950)
![Task](https://img.shields.io/badge/task-scd-9467bd) ![Real](https://img.shields.io/badge/mode-real--training-3fb950)
Binary change detection answers **if** a pixel changed; SCD answers **from what to what**. This notebook uploads (or synthesizes) a pair of land-cover maps, trains a REAL tiny Siamese dual-decoder SCD model in-session, and renders the signature SCD output: two semantic maps plus the **from→to transition matrix** (forest→built-up, farmland→built-up, water→farmland…). This covers the framework's `scd` task (`changemamba_scd`, SECOND-style evaluation) which no other notebook demonstrated.


In [ ]:
#@title ⚙️ SETUP — auto-installs, idempotent {display-mode:"form"}
import subprocess, sys
def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
_pip("pillow", "numpy", "matplotlib")
import os, random, numpy as np, torch
random.seed(42); np.random.seed(42); torch.manual_seed(42)
NOTEBOOK = "scd_colab.ipynb"
OUT_DIR = "outputs"; os.makedirs(OUT_DIR, exist_ok=True)
STATE = {}
print("✅ setup complete — torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
#@title ☁️ MOUNT GOOGLE DRIVE — persistent uploads, checkpoints & results {display-mode:"form"}
MOUNT_DRIVE = True  #@param {type:"boolean"}
DRIVE_PROJECT = "AnyDisasterMapping"  #@param {type:"string"}

# Shared notebook core lives in tools/colab_kit.py (single source of truth).
def _ensure_colab_kit():
    import os as _os, sys as _sys, tempfile as _tf, urllib.request as _uq
    _d = _os.path.abspath(_os.getcwd())
    for _ in range(8):
        _p = _os.path.join(_d, "tools")
        if _os.path.isfile(_os.path.join(_p, "colab_kit.py")):
            if _p not in _sys.path:
                _sys.path.insert(0, _p)
            return "local", _p
        _nd = _os.path.dirname(_d)
        if _nd == _d:
            break
        _d = _nd
    _dest = _os.path.join(_tf.gettempdir(), "adm_colab_kit")
    _os.makedirs(_dest, exist_ok=True)
    _fp = _os.path.join(_dest, "colab_kit.py")
    if not _os.path.isfile(_fp):
        for _rev in ("v5.0.0", "main"):
            _url = ("https://raw.githubusercontent.com/SyedRafiAhmedGIS/"
                    f"AnyDisasterMapping/{_rev}/tools/colab_kit.py")
            try:
                _uq.urlretrieve(_url, _fp)
                if _rev != "v5.0.0":
                    print("⚠️ colab_kit @ v5.0.0 not published yet — fetched 'main'; "
                          "pin activates once the tag is pushed.")
                else:
                    print(f"📥 fetched tools/colab_kit.py @ {_rev}")
                break
            except Exception:
                continue
        else:
            raise RuntimeError("colab_kit.py not found locally and fetch failed — "
                               "run this notebook from the repository checkout")
    if _dest not in _sys.path:
        _sys.path.insert(0, _dest)
    return "fetched", _dest

_ensure_colab_kit()
import colab_kit

DRIVE_ROOT, IN_COLAB = colab_kit.mount_and_prepare(DRIVE_PROJECT, MOUNT_DRIVE)

In [ ]:
#@title 📤 UPLOAD — pre/post land-cover maps · inline preview · synthetic fallback {display-mode:"form"}
import io, os, numpy as np
import matplotlib.pyplot as plt
from PIL import Image

CLASSES = ["nodata", "water", "forest", "farmland", "built-up", "bare"]
N_CLASSES = len(CLASSES)
CMAP = ["#000000", "#1f77b4", "#2ca02c", "#bcbd22", "#d62728", "#d2b48c"]

def read_map():
    print("📂 file picker — choose a CLASS-INDEX map (png; pixel value = class id)")
    try:
        from google.colab import files as _f
        up = _f.upload()
        for nm, data in up.items():
            im = Image.open(io.BytesIO(data)).convert("L").resize((128, 128), Image.NEAREST)
            return np.asarray(im)
    except Exception:
        pass
    return None

pre, post = read_map(), read_map()

if pre is None or post is None:
    print("🛟 missing map(s) — generating a SYNTHETIC land-cover pair with realistic")
    print("   t1→t2 transitions (urban growth onto farmland, river shift, …)")
    rng = np.random.default_rng(42)
    yy, xx = np.mgrid[0:128, 0:128]
    pre = np.zeros((128, 128), np.uint8)
    pre[(yy < 34)] = 1                                   # river band → water
    pre[(yy >= 34) & (xx > 84)] = 2                      # forest east
    pre[(yy >= 34) & (xx <= 84) & (xx > 40)] = 3         # farmland mid
    pre[(yy >= 34) & (xx <= 40)] = 4                     # built-up west
    pre[(yy >= 90) & (xx <= 40)] = 5                     # bare patch
    post = pre.copy()
    growth = (xx <= 84) & (xx > 62) & (yy > 34) & (yy < 100)   # city grows east
    post[growth] = 4
    river_shift = (yy < 30) & (xx > 100)                        # river meanders
    post[river_shift] = 1; post[(yy < 34) & (yy >= 30) & (xx > 100)] = 3
    post[(pre == 4) & (rng.random((128, 128)) < 0.04)] = 5      # scattered demolition
    STATE_SRC = "SYNTHETIC-DEMO"
else:
    STATE_SRC = "USER-UPLOAD"

pre, post = np.clip(pre, 0, N_CLASSES - 1).astype(np.uint8), np.clip(post, 0, N_CLASSES - 1).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), constrained_layout=True)
for ax, m, t_ in zip(axes[:2], [pre, post], ["t1 (pre)", "t2 (post)"]):
    ax.imshow(m, cmap="tab10", vmin=0, vmax=N_CLASSES - 1, interpolation="nearest")
    ax.set_title(t_); ax.axis("off")
change_raw = (pre != post)
axes[2].imshow(change_raw, cmap="hot"); axes[2].set_title("raw change (any class)"); axes[2].axis("off")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in CMAP]
fig.legend(handles, CLASSES, loc="lower center", ncol=6, fontsize=9, bbox_to_anchor=(0.5, -0.06))
fig.suptitle("uploaded / synthetic land-cover pair")
plt.show()
print("source:", STATE_SRC, "| changed pixels: {:.1%}".format(change_raw.mean()))


In [ ]:
#@title ▶️ RUN — tiny Siamese dual-decoder SCD trained in-session (~90s) {display-mode:"form"}
#@markdown Trains a REAL SCD mini-model on the pair (and augmented variants), then predicts semantic maps + the from→to transition matrix. Framework equivalent: `changemamba_scd` / SECOND-style evaluation.

import torch, torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

dev = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

class TinySiameseSCD(nn.Module):
    """Shared encoder, two semantic heads (t1, t2)."""
    def __init__(self, cin=3, classes=N_CLASSES):
        super().__init__()
        def blk(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.ReLU(), nn.Conv2d(o, o, 3, padding=1), nn.ReLU())
        self.enc1, self.enc2, self.enc3 = blk(cin, 16), blk(16, 32), blk(32, 64)
        self.p = nn.MaxPool2d(2)
        self.u2, self.u1 = blk(32 + 16, 16), blk(16, 8)
        self.head1, self.head2 = nn.Conv2d(8, classes, 1), nn.Conv2d(8, classes, 1)
    def forward(self, x):
        s1 = self.enc1(x); s2 = self.enc2(self.p(s1)); b = self.enc3(self.p(s2))
        u2 = self.u2(torch.cat([nn.functional.interpolate(b, scale_factor=2, mode="bilinear"), s2], 1))
        u1 = self.u1(nn.functional.interpolate(u2, scale_factor=2, mode="bilinear"))
        return self.head1(u1), self.head2(u1)

def to_input(m):  # class map → 3-ch tensor (normalized one-hot-ish)
    x = np.zeros((3, 128, 128), np.float32)
    for c in range(1, N_CLASSES):
        x[c % 3][m == c] = 0.5 + 0.1 * c
    return x

net = TinySiameseSCD().to(dev)
opt = torch.optim.Adam(net.parameters(), 1e-3)
x1 = torch.from_numpy(to_input(pre))[None].to(dev)
x2 = torch.from_numpy(to_input(post))[None].to(dev)
y1 = torch.from_numpy(pre.astype(np.int64))[None].to(dev)
y2 = torch.from_numpy(post.astype(np.int64))[None].to(dev)

for epoch in range(60):
    opt.zero_grad()
    h1, h2 = net(x1 + 0.02 * torch.randn_like(x1)), net(x2 + 0.02 * torch.randn_like(x2))
    loss = (nn.functional.cross_entropy(h1, y1) + nn.functional.cross_entropy(h2, y2)) / 2
    loss.backward(); opt.step()
print(f"✅ trained — final loss {loss.item():.4f} on {dev}")

with torch.no_grad():
    p1 = net(x1)[0][0].argmax(0).cpu().numpy()
    p2 = net(x2)[0][0].argmax(0).cpu().numpy()

# -- from→to transition matrix -------------------------------------------------
K = N_CLASSES
T = np.zeros((K, K), np.int64)
for a, b in zip(pre.ravel(), post.ravel()):
    T[a, b] += 1

fig, axes = plt.subplots(1, 4, figsize=(17, 4.2), constrained_layout=True)
axes[0].imshow(pre, cmap="tab10", vmin=0, vmax=K - 1, interpolation="nearest"); axes[0].set_title("t1 reference"); axes[0].axis("off")
axes[1].imshow(p1, cmap="tab10", vmin=0, vmax=K - 1, interpolation="nearest"); axes[1].set_title("SCD pred t1"); axes[1].axis("off")
axes[2].imshow(p2, cmap="tab10", vmin=0, vmax=K - 1, interpolation="nearest"); axes[2].set_title("SCD pred t2"); axes[2].axis("off")
im = axes[3].imshow(np.log1p(T), cmap="viridis"); axes[3].set_title("from→to matrix — reference pair (log)")
axes[3].set_xticks(range(K), CLASSES, rotation=60, fontsize=8); axes[3].set_yticks(range(K), CLASSES, fontsize=8)
plt.colorbar(im, ax=axes[3], fraction=0.046)
fig.suptitle("Semantic Change Detection — semantic maps + transition matrix")
plt.show()

print("\n📊 top transitions (from → to):")
flat = [((CLASSES[i], CLASSES[j]), T[i, j]) for i in range(K) for j in range(K) if T[i, j] > 0 and i != j]
for (a, b), n in sorted(flat, key=lambda kv: -kv[1])[:6]:
    print(f"   {a:>9} → {b:<9} : {n:6d} px  ({n / pre.size:.1%})")
plt.close("all")


In [ ]:
#@title 📦 EXPORT — results zip + Drive copy + auto-download {display-mode:"form"}

# Shared notebook core lives in tools/colab_kit.py (single source of truth).
def _ensure_colab_kit():
    import os as _os, sys as _sys, tempfile as _tf, urllib.request as _uq
    _d = _os.path.abspath(_os.getcwd())
    for _ in range(8):
        _p = _os.path.join(_d, "tools")
        if _os.path.isfile(_os.path.join(_p, "colab_kit.py")):
            if _p not in _sys.path:
                _sys.path.insert(0, _p)
            return "local", _p
        _nd = _os.path.dirname(_d)
        if _nd == _d:
            break
        _d = _nd
    _dest = _os.path.join(_tf.gettempdir(), "adm_colab_kit")
    _os.makedirs(_dest, exist_ok=True)
    _fp = _os.path.join(_dest, "colab_kit.py")
    if not _os.path.isfile(_fp):
        for _rev in ("v5.0.0", "main"):
            _url = ("https://raw.githubusercontent.com/SyedRafiAhmedGIS/"
                    f"AnyDisasterMapping/{_rev}/tools/colab_kit.py")
            try:
                _uq.urlretrieve(_url, _fp)
                if _rev != "v5.0.0":
                    print("⚠️ colab_kit @ v5.0.0 not published yet — fetched 'main'; "
                          "pin activates once the tag is pushed.")
                else:
                    print(f"📥 fetched tools/colab_kit.py @ {_rev}")
                break
            except Exception:
                continue
        else:
            raise RuntimeError("colab_kit.py not found locally and fetch failed — "
                               "run this notebook from the repository checkout")
    if _dest not in _sys.path:
        _sys.path.insert(0, _dest)
    return "fetched", _dest

_ensure_colab_kit()
import colab_kit

def run_export():
    return colab_kit.export_results_zip_hub(globals())

run_export()

## 🗺️ The full notebook collection

| notebook | what it does |
|---|---|
| [`quickstart_colab.ipynb`](quickstart_colab.ipynb) | 5-minute tour of all 13 disaster domains |
| [`dataset_download_colab.ipynb`](dataset_download_colab.ipynb) | access hub for the 24 registered benchmarks |
| [`checkpoint_zoo_colab.ipynb`](checkpoint_zoo_colab.ipynb) | real weights → real inference (or train tiny in-session) |
| [`demo_colab.ipynb`](demo_colab.ipynb) | single-image demo across tasks |
| [`change_detection_colab.ipynb`](change_detection_colab.ipynb) | binary change detection (pre/post) |
| **`scd_colab.ipynb`** | **semantic change detection — land-cover *from→to* matrices** |
| **`evaluation_metrics_colab.ipynb`** | **mIoU · F1 · confusion matrices for any prediction** |
| [`satellite_finetune_colab.ipynb`](satellite_finetune_colab.ipynb) | finetune on your own labeled pairs |
| [`batch_inference_colab.ipynb`](batch_inference_colab.ipynb) | folder-level batch inference |
| domain notebooks (flood · wildfire · tsunami · hurricane · landslide · volcano) | per-domain guided demos |

Framework: [`SyedRafiAhmedGIS/AnyDisasterMapping`](https://github.com/SyedRafiAhmedGIS/AnyDisasterMapping) — paper: *Earth Observation for Disaster Mapping: Benchmarks, Methods, Challenges and Future Perspectives* (SSRN 6725082).
